# utils_StudentB.ipynb

This notebook documents and demonstrates the functions in `utils.py` written by **Student B**:

1. `fit_tfidf_vectorizer` – fits a TF-IDF model on training questions
2. `cosine_similarity_sparse` – cosine similarity between sparse TF-IDF vectors (from scratch)
3. `get_tfidf_cosine_features` – applies cosine similarity over a DataFrame

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
import pandas as pd
import numpy as np
from utils import (
    get_data_splits,
    fit_tfidf_vectorizer,
    cosine_similarity_sparse,
    get_tfidf_cosine_features,
)

## `cosine_similarity_sparse` – from-scratch implementation

Cosine similarity measures the angle between two vectors:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

Unlike Jaccard, it takes into account **term frequencies** (via TF-IDF weights), so rarer, more informative words contribute more to the similarity score.

The implementation works directly on **scipy sparse row vectors** (no sklearn `cosine_similarity` helper):
- dot product → `v1.dot(v2.T)`
- norms → `sqrt(v.dot(v.T))`

Handles the zero-vector edge case (returns 0.0).

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

mini_tfidf = TfidfVectorizer()
mini_tfidf.fit(["What is machine learning", "How does machine learning work", "What is AI"])

v1 = mini_tfidf.transform(["What is machine learning"])
v2 = mini_tfidf.transform(["What is machine learning"])   # identical
v3 = mini_tfidf.transform(["How does machine learning work"])  # similar
v4 = mini_tfidf.transform(["What is the weather today"])   # different

print("identical questions:", round(cosine_similarity_sparse(v1, v2), 4))  # 1.0
print("similar questions:  ", round(cosine_similarity_sparse(v1, v3), 4))  # > 0
print("different questions:", round(cosine_similarity_sparse(v1, v4), 4))  # ~0

identical questions: 1.0
similar questions:   0.373
different questions: 0.7071


## `fit_tfidf_vectorizer` & `get_tfidf_cosine_features`

- `fit_tfidf_vectorizer` trains on all questions in the **training split only** (no data leakage).
- `get_tfidf_cosine_features` vectorises each question pair and calls `cosine_similarity_sparse` for every row, returning a column of shape `(n, 1)`.

In [3]:
DATA_PATH = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)
train_df, val_df, test_df = get_data_splits(quora_df)

tfidf_vec = fit_tfidf_vectorizer(train_df)
print("TF-IDF vocab size:", len(tfidf_vec.vocabulary_))

TF-IDF vocab size: 50000


In [4]:
sample = train_df.sample(500, random_state=42)
X_cos = get_tfidf_cosine_features(sample, tfidf_vec)

print("TF-IDF cosine features shape:", X_cos.shape)

dup_mask = sample["is_duplicate"].values.astype(bool)
print(f"\n  mean cosine for duplicates     = {X_cos[dup_mask].mean():.3f}")
print(f"  mean cosine for non-duplicates = {X_cos[~dup_mask].mean():.3f}")

TF-IDF cosine features shape: (500, 1)

  mean cosine for duplicates     = 0.477
  mean cosine for non-duplicates = 0.328


The mean cosine similarity is substantially higher for duplicate pairs, confirming the feature discriminates well between classes.

### Why TF-IDF cosine over plain Jaccard?
TF-IDF downweights very common words (like *"what"*, *"how"*, *"the"*) that appear in nearly every question but carry little semantic meaning. Two questions sharing the word *"photosynthesis"* should be penalised less than two questions both containing *"what"*. TF-IDF handles this automatically, while Jaccard treats all tokens equally.